## Business Insight

### What business_insight.py handle ?
* Calculates financial damage
* Classifies operational risk
* Suggests maintenance actions
* Identifies peak stress hours

This function is actually the business intelligence brain of your Energy AI project.

It converts raw ML anomaly outputs into:

* Financial impact

* Risk severity

* Peak anomaly timing

* Smart maintenance recommendations


#### Information : 

* The generate_business_insights() function converts anomaly detection results into actionable business intelligence.
*  It filters data based on user selection, calculates peak anomaly periods, estimates financial impact, classifies severity levels, and dynamically generates maintenance recommendations based on operational risk.

## Function

In [ ]:
def generate_business_insights(....)

This function:

* Filters data (month/date)

* Calculates anomaly statistics

* Estimates financial loss

* Classifies severity level

* Generates automated business recommendations

* Returns structured insights dictionary

* It transforms technical ML output → executive-level insights

## Month Filter

In [ ]:
    if selected_month is not None:
        df_filtered = df_filtered[df_filtered["month"] == selected_month]

If user selects:

Month = 5 (May)

Then only May data is analyzed.

This allows:

* Monthly anomaly study

* Seasonal performance comparison

* Executive filtering in dashboard

## Date Filter

In [ ]:
 if selected_date is not None:
        df_filtered = df_filtered[
            df_filtered["timestamp"].dt.date == selected_date
        ]

Filters for a specific day.

Example:

Analyze anomalies on 2024-01-15 only.

Used for:

* Incident investigation

* Daily spike analysis

* Energy audit day tracking

## Peak Hour

In [ ]:
    peak_hours = (
        df_filtered[df_filtered["final_anomaly"] == 1]
        ["hour"]
        .value_counts()
        .sort_index()
    )

What happens:

* Select only anomaly rows

* Take "hour" column

* Count how many anomalies per hour

* Sort chronologically


## Peak Month Calculation

This helps detect:

* Seasonal stress

* Counts anomalies per month.

* Summer cooling overload

* Winter heating overload

Note: This uses full dataset, not filtered one


## Cost Impact Calculation

In [ ]:
energy_col = df.select_dtypes(include="number").columns[0]

Automatically selects first numeric column (usually electricity usage).

In [ ]:
anomaly_energy = df_filtered[
    df_filtered["final_anomaly"] == 1
][energy_col].sum()

Adds total energy consumed during anomaly events.

In [ ]:
estimated_cost = anomaly_energy * cost_per_unit

Example:

If:

100,000 kWh wasted

Cost per unit = $0.12

Then:

Estimated Loss = $12,000

This converts anomaly detection → real financial damage.

## Anomaly Rate

In [ ]:
anomaly_rate = round(
    df_filtered["final_anomaly"].mean() * 100, 2
)

Since anomaly is 1 or 0:

Mean = anomaly ratio.

Example:

0.085 → 8.5%

## Severity Classification

Business-friendly classification:

Rate	Severity
* =>15%	High
* 7–15% 	Medium
* <7%	Low

This makes output executive-ready.

## Recommendation Engine

Dynamic recommendations based on:

A) Severity

If HIGH:

Immediate inspection

HVAC diagnostics

Real-time monitoring

If MEDIUM:

Preventive maintenance

Schedule optimization

If LOW:

Continue monitoring

B) Cost-Based Suggestions
if estimated_cost > 10000:

If financial impact high → prioritize strategy.

C) Peak-Hour Suggestion
peak_hour = peak_hours.idxmax()

Find hour with maximum anomalies.

Adds:

Peak anomalies frequently occur around 14:00
Inspect load conditions during this time.

This is powerful because:

* ML finds anomaly
* This finds operational cause window

## Final Insights Dictionary

In [ ]:
insights = {
    ...
}

## Recommendation
{
  "estimated_cost_loss_$": 12000,
  "anomaly_rate_percent": 8.5,
  "total_anomalies": 4000,
  "severity_level": "Medium",
  "peak_anomaly_months": {...},
  "peak_hours": {...},
  "recommendations": [...]
}

This is later used in:

Streamlit dashboard

PDF report

Executive presentatio

## Whole Code

In [1]:
def generate_business_insights(
    df,
    cost_per_unit=0.12,
    selected_month=None,
    selected_date=None
):

    df_filtered = df.copy()
    
    
    # -------------------------------------------------
    # MONTH FILTER
    # -------------------------------------------------
    if selected_month is not None:
        df_filtered = df_filtered[df_filtered["month"] == selected_month]

    # -------------------------------------------------
    # DATE FILTER
    # -------------------------------------------------
    if selected_date is not None:
        df_filtered = df_filtered[
            df_filtered["timestamp"].dt.date == selected_date
        ]

    # -------------------------------------------------
    # PEAK HOUR CALCULATION
    # -------------------------------------------------
    peak_hours = (
        df_filtered[df_filtered["final_anomaly"] == 1]
        ["hour"]
        .value_counts()
        .sort_index()
    )
    # -------------------------------------------------
    # PEAK months CALCULATION
    # -------------------------------------------------
    peak_months = df[df["final_anomaly"]==1]["month"].value_counts().to_dict()
    # -------------------------------------------------
    # COST IMPACT
    # -------------------------------------------------
    energy_col = df.select_dtypes(include="number").columns[0]

    anomaly_energy = df_filtered[
        df_filtered["final_anomaly"] == 1
    ][energy_col].sum()

    estimated_cost = round(anomaly_energy * cost_per_unit, 2)

    anomaly_rate = round(
        df_filtered["final_anomaly"].mean() * 100, 2
    )

    total_anomalies = int(df_filtered["final_anomaly"].sum())

    # -------------------------------------------------
    # SEVERITY CLASSIFICATION
    # -------------------------------------------------
    if anomaly_rate > 15:
        severity = "High"
    elif anomaly_rate > 7:
        severity = "Medium"
    else:
        severity = "Low"

    # -------------------------------------------------
    # RECOMMENDATION ENGINE
    # -------------------------------------------------
    recommendations = []

    if severity == "High":
        recommendations.append(
            " Immediate equipment inspection required. High anomaly rate detected."
        )
        recommendations.append(
            " Conduct HVAC ( Heat,Vantilation,Air,Conditining ) and electrical load diagnostics."
        )
        recommendations.append(
            " Implement real-time monitoring alerts to prevent recurring spikes."
        )

    elif severity == "Medium":
        recommendations.append(
            "⚠ Schedule preventive maintenance within this month."
        )
        recommendations.append(
            " Analyze operational schedules for inefficiencies."
        )

    else:
        recommendations.append(
            " System operating within acceptable anomaly range."
        )
        recommendations.append(
            " Continue monitoring for early detection of faults."
        )

    # Cost-based suggestion
    if estimated_cost > 10000:
        recommendations.append(
            " High financial impact detected. Prioritize anomaly reduction strategy."
        )
    elif estimated_cost > 2000:
        recommendations.append(
            " Moderate cost impact. Optimize peak-hour operations."
        )

    # Peak-hour-based suggestion
    if len(peak_hours) > 0:
        peak_hour = peak_hours.idxmax()
        recommendations.append(
            f" Peak anomalies frequently occur around {int(peak_hour):02d}:00. "
            "Inspect load conditions during this time window."
        )

    # -------------------------------------------------
    # INSIGHTS DICTIONARY
    # -------------------------------------------------
    insights = {
        "estimated_cost_loss_$": estimated_cost,
        "anomaly_rate_percent": anomaly_rate,
        "total_anomalies": total_anomalies,
        "severity_level": severity,
        "peak_anomaly_months": peak_months,
        "peak_hours": peak_hours,
        "recommendations": recommendations
    }

    return insights, df_filtered